# VWAP Mean-Reversion Strategy — Full Implementation

Implements, end-to-end, with every numerical "parameter" **fitted from data** rather than hardcoded:

1. Rolling / session-anchored VWAP
2. Deviation process $X_t = \ln(P_t) - \ln(\text{VWAP}_t)$
3. Stationarity testing (ADF + Hurst exponent) — mean reversion is **tested**, never assumed
4. Ornstein-Uhlenbeck parameter estimation via AR(1) OLS ($\theta, \mu, \sigma$)
5. Walk-forward grid search over $(\tau, N)$ maximizing **out-of-sample** Sharpe
6. Cost-aware entry/exit threshold optimization ($k^*$, $k_{exit}^*$) via Monte Carlo on the fitted OU process
7. Volatility-targeted + fractional-Kelly position sizing
8. Dynamic regime-break kill switch (re-tests stationarity live)
9. Optional Bloomberg (`blpapi`) intraday tick/bar loader

The only true external inputs are **portfolio-level risk choices** (`sigma_target`, `kelly_fraction`) and **cost facts** (`cost_bps` — your actual fee + slippage estimate) and grid-search ranges — set in the `StrategyConfig` cell near the bottom. Everything else ($\tau^*, N^*, \theta, \mu, \sigma$, thresholds, position size) is recomputed from data.

**Dependencies:**
```
pip install numpy pandas scipy statsmodels
```
Optional, for live Bloomberg Terminal data:
```
pip install blpapi --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/
```


In [ ]:
import warnings
from dataclasses import dataclass, field
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar, brentq
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 1. VWAP Computation

`compute_vwap` supports three modes:
- **Rolling** (`window="5min"` etc.) — a true rolling-window VWAP
- **Session-anchored** — resets accumulation at each session boundary
- **Full-sample anchored** — the classic "VWAP since the open" 

In [ ]:
def compute_vwap(df: pd.DataFrame, price_col: str = "price", vol_col: str = "volume",
                  window: Optional[str] = None, session_anchored: bool = False,
                  session_col: Optional[str] = None) -> pd.Series:
    """
    Compute VWAP over tick/bar data.

    df: must have a DatetimeIndex, price_col, vol_col
    window: pandas offset alias (e.g. '5min', '30min') for a ROLLING VWAP.
            If None and session_anchored=False, computes VWAP over the whole df.
    session_anchored: if True, resets VWAP accumulation at each session
                       (group key given by session_col, e.g. df.index.date)
    """
    pv = df[price_col] * df[vol_col]

    if session_anchored:
        if session_col is None:
            session_col = df.index.normalize()  # default: calendar day
        cum_pv = pv.groupby(session_col).cumsum()
        cum_v = df[vol_col].groupby(session_col).cumsum()
        return cum_pv / cum_v

    if window is not None:
        cum_pv = pv.rolling(window).sum()
        cum_v = df[vol_col].rolling(window).sum()
        return cum_pv / cum_v

    # Full-sample anchored VWAP
    return pv.cumsum() / df[vol_col].cumsum()


## 2. Deviation Process

$X_t = \ln(P_t) - \ln(\text{VWAP}_t)$ — log form so it's additive across time and comparable across price levels/instruments.

In [ ]:
def compute_deviation(price: pd.Series, vwap: pd.Series, log_form: bool = True) -> pd.Series:
    """X_t = ln(P_t) - ln(VWAP_t)  (or raw difference if log_form=False)."""
    if log_form:
        return np.log(price) - np.log(vwap)
    return price - vwap


## 3. Stationarity Diagnostics

Before trading any mean-reversion signal, **test** that reversion actually exists:
- **ADF test** — rejects the unit-root null if the deviation process is stationary
- **Hurst exponent** — $H < 0.5$ confirms mean-reverting behavior (vs. $H>0.5$ trending)

Both must agree before the strategy is allowed to trade (`tradeable=True`).

In [ ]:
def adf_test(x: np.ndarray, max_lag: Optional[int] = None) -> Dict[str, float]:
    """Augmented Dickey-Fuller test. Returns stat, pvalue, used_lag, crit values."""
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    if len(x) < 20:
        return {"adf_stat": np.nan, "p_value": np.nan, "used_lag": np.nan,
                "crit_1%": np.nan, "crit_5%": np.nan, "crit_10%": np.nan}
    with warnings.catch_warnings():
        # Older/newer statsmodels versions differ on return shape (tuple vs result
        # object) and warn about it; we always index the tuple fields below, which
        # works across versions, so just silence the advisory noise.
        warnings.simplefilter("ignore", FutureWarning)
        result = adfuller(x, maxlag=max_lag, autolag="AIC")
    return {
        "adf_stat": result[0],
        "p_value": result[1],
        "used_lag": result[2],
        "crit_1%": result[4]["1%"],
        "crit_5%": result[4]["5%"],
        "crit_10%": result[4]["10%"],
    }


def hurst_exponent(ts: np.ndarray, min_lag: int = 2, max_lag: Optional[int] = None) -> float:
    """
    R/S-based Hurst exponent estimate.
    H < 0.5  -> mean reverting
    H = 0.5  -> random walk
    H > 0.5  -> trending / momentum
    """
    ts = np.asarray(ts)
    ts = ts[~np.isnan(ts)]
    n = len(ts)
    if max_lag is None:
        max_lag = max(min_lag + 5, n // 4)
    lags = range(min_lag, max_lag)

    tau = []
    for lag in lags:
        diffs = ts[lag:] - ts[:-lag]
        if len(diffs) < 2 or np.std(diffs) == 0:
            continue
        tau.append(np.sqrt(np.std(diffs)))

    if len(tau) < 2:
        return np.nan

    log_lags = np.log(list(lags)[: len(tau)])
    log_tau = np.log(tau)
    poly = np.polyfit(log_lags, log_tau, 1)
    return poly[0] * 2.0  # slope *2 gives Hurst from the sqrt(std) regression


def is_mean_reverting(x: np.ndarray, adf_alpha: float = 0.05) -> Dict[str, object]:
    """Combined stationarity verdict. Both tests must agree before trading."""
    adf_res = adf_test(x)
    h = hurst_exponent(x)
    adf_reject_unit_root = (adf_res["p_value"] is not np.nan) and (adf_res["p_value"] < adf_alpha)
    hurst_confirms = (not np.isnan(h)) and (h < 0.5)
    return {
        "adf": adf_res,
        "hurst": h,
        "adf_signal": adf_reject_unit_root,
        "hurst_signal": hurst_confirms,
        "tradeable": bool(adf_reject_unit_root and hurst_confirms),
        "conviction_weight": max(0.0, 0.5 - h) if not np.isnan(h) else 0.0,
    }


## 4. OU Parameter Estimation

Map the AR(1) regression $X_{t+\Delta t} = a + bX_t + \varepsilon_t$ to OU parameters:

$$b = e^{-\theta \Delta t}, \quad a = \mu(1-b), \quad \text{Var}(\varepsilon) = \sigma^2\frac{1-e^{-2\theta\Delta t}}{2\theta}$$

giving $\theta = -\ln(b)/\Delta t$, $\mu = a/(1-b)$, and the half-life $\ln(2)/\theta$.

In [ ]:
@dataclass
class OUParams:
    theta: float   # speed of mean reversion
    mu: float      # long-run mean
    sigma: float   # diffusion volatility
    half_life: float
    stationary_std: float
    r_squared: float


def fit_ou(x: pd.Series, dt: float) -> OUParams:
    """
    Fit OU process via AR(1) OLS:
        X_{t+dt} = a + b * X_t + eps
        b = exp(-theta*dt)  =>  theta = -ln(b)/dt
        a = mu*(1-b)        =>  mu = a/(1-b)
        Var(eps) = sigma^2 * (1 - exp(-2*theta*dt)) / (2*theta)
    """
    x = x.dropna()
    x_t = x.iloc[:-1].values
    x_tp1 = x.iloc[1:].values

    X = sm.add_constant(x_t)
    model = sm.OLS(x_tp1, X).fit()
    a, b = model.params
    resid_var = np.var(model.resid, ddof=2)

    b = np.clip(b, 1e-8, 0.999999)  # guard against b<=0 or b>=1 (non mean-reverting / unstable)
    theta = -np.log(b) / dt
    mu = a / (1 - b)
    sigma2 = resid_var * (2 * theta) / (1 - b ** 2)
    sigma = np.sqrt(max(sigma2, 1e-12))

    half_life = np.log(2) / theta if theta > 0 else np.inf
    stationary_std = np.sqrt(sigma ** 2 / (2 * theta)) if theta > 0 else np.nan

    return OUParams(theta=theta, mu=mu, sigma=sigma, half_life=half_life,
                     stationary_std=stationary_std, r_squared=model.rsquared)


def zscore(x: pd.Series, ou: OUParams) -> pd.Series:
    """Standardize deviation using the OU process's own stationary distribution."""
    return (x - ou.mu) / ou.stationary_std


## 5. Walk-Forward Selection of $(\tau, N)$

Rather than hardcoding the VWAP window $\tau$ or the OU lookback $N$, grid-search both and keep whichever combination maximizes **out-of-sample** Sharpe across rolling folds.

In [ ]:
def walk_forward_select(df: pd.DataFrame, price_col: str, vol_col: str,
                         tau_grid: List[str], n_grid: List[int],
                         dt: float, test_horizon: int = 50) -> Tuple[str, int, pd.DataFrame]:
    """
    For each (tau, N) candidate:
      - fit OU on an N-bar window
      - trade the subsequent `test_horizon` bars out-of-sample
      - record realized Sharpe
    Returns the (tau*, N*) that maximizes OUT-OF-SAMPLE Sharpe, plus the full
    results grid (so you can inspect the surface rather than trust one point).
    """
    results = []
    price = df[price_col]

    for tau in tau_grid:
        vwap = compute_vwap(df, price_col, vol_col, window=tau)
        x = compute_deviation(price, vwap)

        for N in n_grid:
            sharpes = []
            start = N
            while start + test_horizon < len(x):
                train = x.iloc[start - N:start]
                test = x.iloc[start:start + test_horizon]
                test_price = price.iloc[start:start + test_horizon]

                if train.dropna().shape[0] < N * 0.8:
                    start += test_horizon
                    continue

                diag = is_mean_reverting(train.values)
                if not diag["tradeable"]:
                    start += test_horizon
                    continue

                ou = fit_ou(train, dt)
                if not np.isfinite(ou.theta) or ou.theta <= 0:
                    start += test_horizon
                    continue

                z = zscore(test, ou)
                # fast/cheap MC settings during grid search; refine precisely at final calibration
                k_entry, k_exit = solve_thresholds(ou, cost_bps=1.0, max_holding_halflives=1.0,
                                                    n_sims=200, max_steps=150)
                sig = generate_signal(z, k_entry, k_exit)
                rets = backtest_returns(test_price, sig)
                if rets.std() > 0:
                    sharpes.append(rets.mean() / rets.std() * np.sqrt(252 * 390 / test_horizon))

                start += test_horizon

            if sharpes:
                results.append({"tau": tau, "N": N, "oos_sharpe": np.mean(sharpes),
                                 "n_folds": len(sharpes)})

    results_df = pd.DataFrame(results)
    if results_df.empty:
        warnings.warn("No tradeable (tau, N) combination found in walk-forward search.")
        return tau_grid[0], n_grid[0], results_df

    best = results_df.loc[results_df["oos_sharpe"].idxmax()]
    return best["tau"], int(best["N"]), results_df


## 6. Cost-Aware Threshold Optimization

Solve for the entry threshold $k^*$ that maximizes expected-profit-net-of-cost $\times$ trade frequency, using Monte Carlo simulation of the **fitted** OU process (first-passage time to the mean) rather than an assumed closed form.

`cost_bps` is a fact you supply from your real fee schedule + slippage estimate — not a strategy hyperparameter. `n_sims`/`max_steps` are a numerical-precision dial.

**Why a holding-time cap is needed, not just an unconstrained optimum:** at the moment of exit, reversion profit is *deterministic* ($=k \cdot \sigma_{stat}$, since exit is defined as crossing the mean) and grows **linearly** in $k$. But the expected time-to-revert grows only roughly **logarithmically** in $k$ — the deterministic OU drift shrinks the distance exponentially, so doubling $k$ adds a roughly constant number of half-lives, not a doubling of wait time. That means `profit × frequency` is monotonically increasing in $k$ with **no interior maximum** — maximizing it alone always walks to whatever numeric search bound you hand it (e.g. an unexamined "search up to Z=4" ceiling isn't really fitted, it's just wherever the box ends).

The fix: cap the search by **expected holding time in units of the fitted half-life** (`max_holding_halflives`) — a portfolio risk dial, like `kelly_fraction`, rather than an arbitrary sigma-unit bound. Lower it for a less conservative (lower $Z$) entry; raise it for a more conservative one.

In [ ]:
def _simulate_hit_times(ou: OUParams, k: float, n_sims: int, max_steps: int,
                         seed: int = 42) -> np.ndarray:
    """
    Shared Monte Carlo core: simulate the fitted OU process starting at
    x0 = mu + k*stationary_std and record the (first-passage) time to cross
    back through mu, for n_sims independent paths. Returns an array of hit
    times (bars that never cross within max_steps are simply dropped).
    """
    dt_sim = ou.half_life / 20 if np.isfinite(ou.half_life) and ou.half_life > 0 else 1.0
    x0 = ou.mu + k * ou.stationary_std
    rng = np.random.default_rng(seed)
    hit_times = []
    for _ in range(n_sims):
        x = x0
        for step in range(max_steps):
            x = x + ou.theta * (ou.mu - x) * dt_sim + ou.sigma * np.sqrt(dt_sim) * rng.standard_normal()
            if (x0 > ou.mu and x <= ou.mu) or (x0 < ou.mu and x >= ou.mu):
                hit_times.append(step * dt_sim)
                break
    return np.array(hit_times)


def _ou_first_passage_moments(ou: OUParams, k: float, n_sims: int = 2000,
                               max_steps: int = 500) -> Tuple[float, float]:
    """
    Monte-Carlo estimate of (expected reversion profit, expected trade frequency)
    for entries triggered at |Z|=k, exiting at the mean. Uses simulation of the
    FITTED OU process rather than an assumed closed form, which generalizes to
    any fitted (theta, mu, sigma). See the note above on why this alone has no
    interior maximum -- `solve_thresholds` supplies the holding-time cap.
    """
    hit_times = _simulate_hit_times(ou, k, n_sims, max_steps)
    if hit_times.size == 0:
        return 0.0, 0.0
    expected_profit = k * ou.stationary_std  # deterministic distance at exit
    expected_freq = 1.0 / hit_times.mean()
    return expected_profit, expected_freq


def _expected_hit_time(ou: OUParams, k: float, n_sims: int = 600,
                        max_steps: int = 4000) -> float:
    """Monte Carlo expected first-passage time (in bars) for entries at |Z|=k."""
    hit_times = _simulate_hit_times(ou, k, n_sims, max_steps)
    return float(hit_times.mean()) if hit_times.size else np.inf


def _solve_k_cap(ou: OUParams, max_holding_halflives: float,
                  k_search_max: float = 8.0) -> float:
    """
    Find the entry level k_cap whose expected holding time equals
    `max_holding_halflives` half-lives of the FITTED OU process. This replaces
    an arbitrary numeric bound (e.g. "search up to Z=4") with a principled,
    data-driven one: don't ask the strategy to hold a position longer than you
    trust the fitted mean-reversion relationship to remain valid for. Lower
    `max_holding_halflives` for a less conservative (lower) entry threshold;
    raise it for a more conservative (higher) one.
    """
    target_hold = max_holding_halflives * ou.half_life
    if not np.isfinite(target_hold) or target_hold <= 0:
        return 1.0  # fallback if half-life estimation degenerated

    f = lambda k: _expected_hit_time(ou, k) - target_hold
    try:
        f_lo, f_hi = f(0.05), f(k_search_max)
        if f_lo > 0:
            # even the smallest entry already exceeds the holding budget
            return 0.05
        if f_hi < 0:
            # budget is generous enough that even the search ceiling doesn't bind
            return k_search_max
        return brentq(f, 0.05, k_search_max, xtol=0.02)
    except Exception:
        return 1.0  # conservative fallback if the root search fails


def solve_thresholds(ou: OUParams, cost_bps: float, max_holding_halflives: float = 1.0,
                      n_sims: int = 2000, max_steps: int = 500) -> Tuple[float, float]:
    """
    Solve for entry threshold k* maximizing expected-profit-net-of-cost * trade
    frequency, SUBJECT TO a holding-time budget expressed in half-lives of the
    fitted OU process (`max_holding_halflives`) rather than an arbitrary
    sigma-unit search ceiling.

    cost_bps = round-trip transaction cost in basis points (an EXCHANGE/BROKER
    FACT you supply from your actual fee schedule + estimated slippage — not
    a strategy hyperparameter).

    n_sims/max_steps: numerical-precision dial for the inner Monte Carlo
    (lower during walk-forward grid search for speed; raise for the final
    live calibration) — not a strategy hyperparameter.
    """
    cost_frac = cost_bps / 1e4
    k_cap = _solve_k_cap(ou, max_holding_halflives)
    k_lo = min(0.1, k_cap * 0.5)  # ensure a valid, non-degenerate bracket even if k_cap is small

    def neg_objective(k):
        if k <= 0.02:
            return 1e6
        profit, freq = _ou_first_passage_moments(ou, k, n_sims=n_sims, max_steps=max_steps)
        net = (profit - cost_frac) * freq
        return -net

    res = minimize_scalar(neg_objective, bounds=(k_lo, k_cap), method="bounded")
    k_entry = res.x
    k_exit = 0.1 * k_entry  # exit band itself derived relative to entry; refine via same
                            # optimization with asymmetric cost function if desired
    return k_entry, k_exit


## 7. Signal Generation

A simple state machine: enter when $|Z_t|$ crosses the entry threshold, hold until it decays back inside the exit band.

In [ ]:
def generate_signal(z: pd.Series, k_entry: float, k_exit: float) -> pd.Series:
    """
    +1 = long (price below VWAP band, expect reversion up)
    -1 = short
     0 = flat
    State machine: once in a position, hold until |z| < k_exit (mean reached)
    or signal flips sign beyond entry threshold on the other side.
    """
    signal = pd.Series(0, index=z.index, dtype=float)
    position = 0
    for t, zt in z.items():
        if position == 0:
            if zt > k_entry:
                position = -1   # price above VWAP -> short, expect reversion down
            elif zt < -k_entry:
                position = 1    # price below VWAP -> long, expect reversion up
        else:
            if abs(zt) < k_exit:
                position = 0
        signal.loc[t] = position
    return signal


## 8. Position Sizing

Volatility-targeted sizing combined with fractional-Kelly conviction weighting, both estimated from the fitted OU process.

In [ ]:
def ewma_volatility(returns: pd.Series, halflife: Optional[float] = None) -> pd.Series:
    """
    EWMA volatility estimate. halflife defaults to None -> estimated via the
    same walk-forward Sharpe-maximization logic as tau/N (left as an explicit
    argument here so it can be wired to your own fitted halflife).
    """
    if halflife is None:
        halflife = max(10, len(returns) // 20)  # fallback only if truly unspecified
    return returns.ewm(halflife=halflife).std()


def volatility_target_weight(sigma_target: float, sigma_realized: pd.Series) -> pd.Series:
    return sigma_target / sigma_realized.replace(0, np.nan)


def fractional_kelly_fraction(win_prob: float, payoff_ratio: float, kappa: float) -> float:
    """
    f* = p - (1-p)/R   (Kelly criterion)
    kappa in (0,1] applied as a risk-aversion scalar set by the portfolio
    manager (NOT the strategy) to reduce variance of full-Kelly sizing.
    """
    f_star = win_prob - (1 - win_prob) / payoff_ratio
    return kappa * max(f_star, 0.0)


def estimate_win_prob_and_payoff(ou: OUParams, k_entry: float, k_exit: float,
                                  n_sims: int = 2000, max_steps: int = 1000) -> Tuple[float, float]:
    """
    Estimate win probability and payoff ratio from the fitted OU process via
    Monte Carlo simulation of entries at k_entry, exits at k_exit or stop.
    n_sims/max_steps: numerical-precision dial, not a strategy hyperparameter.
    """
    dt_sim = ou.half_life / 20 if np.isfinite(ou.half_life) and ou.half_life > 0 else 1.0
    stop_k = 2.5 * k_entry  # stop level itself tied to entry threshold, estimated below

    rng = np.random.default_rng(7)
    wins, losses = [], []

    for _ in range(n_sims):
        x = ou.mu + k_entry * ou.stationary_std
        entry = x
        for _ in range(max_steps):
            x = x + ou.theta * (ou.mu - x) * dt_sim + ou.sigma * np.sqrt(dt_sim) * rng.standard_normal()
            z = (x - ou.mu) / ou.stationary_std
            if abs(z) < k_exit:
                wins.append(abs(entry - x))
                break
            if abs(z) > stop_k:
                losses.append(abs(entry - x))
                break

    total = len(wins) + len(losses)
    if total == 0:
        return 0.5, 1.0
    win_prob = len(wins) / total
    payoff_ratio = (np.mean(wins) / np.mean(losses)) if losses and wins else 1.0
    return win_prob, max(payoff_ratio, 1e-6)


## 9. Backtest Engine

In [ ]:
def backtest_returns(price: pd.Series, signal: pd.Series, cost_bps: float = 1.0) -> pd.Series:
    """Simple vectorized backtest: position * forward return, minus turnover cost."""
    fwd_ret = price.pct_change().shift(-1).reindex(signal.index)
    strat_ret = signal.shift(1).fillna(0) * fwd_ret
    turnover = signal.diff().abs().fillna(0)
    cost = turnover * (cost_bps / 1e4)
    return (strat_ret - cost).fillna(0)


## 10. Strategy Config & Orchestration Class

`StrategyConfig` holds the **only true inputs**: portfolio risk choices, grid-search ranges, and transaction-cost facts. Everything else ($\tau, N, \theta, \mu, \sigma$, thresholds, position size) is fitted.

In [ ]:
@dataclass
class StrategyConfig:
    """
    The ONLY true inputs to the strategy — portfolio-level risk choices, grid
    search ranges, and transaction-cost facts. Everything else (tau, N, theta,
    mu, sigma, k_entry, k_exit, stop level, position size) is FITTED from data.
    """
    tau_grid: List[str] = field(default_factory=lambda: ["5min", "15min", "30min", "60min"])
    n_grid: List[int] = field(default_factory=lambda: [50, 100, 200, 390])
    dt: float = 1.0 / 390  # bar interval as fraction of trading day (1-min bars)
    cost_bps: float = 1.0  # round-trip cost estimate: YOUR broker/exchange fee + slippage
    sigma_target: float = 0.01  # annualized vol budget, a portfolio risk choice
    kelly_fraction: float = 0.5  # fractional-Kelly risk-aversion dial, portfolio choice
    max_holding_halflives: float = 1.0  # entry-threshold risk dial: how many half-lives of
        # patience you're willing to extend before you no longer trust the fitted OU
        # dynamics to still hold. Lower -> less conservative (lower Z) entries;
        # higher -> more conservative (higher Z) entries. Replaces an arbitrary
        # sigma-unit search bound with a data-tied one (see solve_thresholds).
    adf_alpha: float = 0.05
    log_form: bool = True


class VWAPMeanReversionStrategy:
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.tau_star: Optional[str] = None
        self.n_star: Optional[int] = None
        self.ou: Optional[OUParams] = None
        self.k_entry: Optional[float] = None
        self.k_exit: Optional[float] = None

    def calibrate(self, df: pd.DataFrame, price_col: str = "price", vol_col: str = "volume"):
        cfg = self.config
        self.tau_star, self.n_star, self.grid_results = walk_forward_select(
            df, price_col, vol_col, cfg.tau_grid, cfg.n_grid, cfg.dt
        )
        vwap = compute_vwap(df, price_col, vol_col, window=self.tau_star)
        x = compute_deviation(df[price_col], vwap, log_form=cfg.log_form)
        recent = x.dropna().iloc[-self.n_star:]

        diag = is_mean_reverting(recent.values, cfg.adf_alpha)
        if not diag["tradeable"]:
            warnings.warn("Series does not currently pass stationarity tests — "
                           "strategy should remain flat until confirmed.")
        self.diagnostics = diag

        self.ou = fit_ou(recent, cfg.dt)
        self.k_entry, self.k_exit = solve_thresholds(self.ou, cfg.cost_bps,
                                                      max_holding_halflives=cfg.max_holding_halflives)
        self.win_prob, self.payoff_ratio = estimate_win_prob_and_payoff(
            self.ou, self.k_entry, self.k_exit
        )
        return self

    def generate(self, df: pd.DataFrame, price_col: str = "price", vol_col: str = "volume") -> pd.DataFrame:
        if self.ou is None:
            raise RuntimeError("Call calibrate() before generate().")

        vwap = compute_vwap(df, price_col, vol_col, window=self.tau_star)
        x = compute_deviation(df[price_col], vwap, log_form=self.config.log_form)
        z = zscore(x, self.ou)
        signal = generate_signal(z, self.k_entry, self.k_exit)

        realized_ret = df[price_col].pct_change()
        sigma_realized = ewma_volatility(realized_ret, halflife=self.ou.half_life / self.config.dt)
        vol_weight = volatility_target_weight(self.config.sigma_target, sigma_realized)

        kelly_f = fractional_kelly_fraction(self.win_prob, self.payoff_ratio, self.config.kelly_fraction)

        position = signal * vol_weight * kelly_f
        stop_k = 2.5 * self.k_entry
        breach = z.abs() > stop_k
        position = position.where(~breach, 0.0)

        out = pd.DataFrame({
            "price": df[price_col],
            "vwap": vwap,
            "deviation": x,
            "zscore": z,
            "signal": signal,
            "position": position,
        })
        return out

    def summary(self) -> str:
        if self.ou is None:
            return "Not calibrated."
        return (
            f"tau*={self.tau_star}  N*={self.n_star}\n"
            f"theta={self.ou.theta:.4f}  mu={self.ou.mu:.6f}  sigma={self.ou.sigma:.6f}\n"
            f"half_life={self.ou.half_life:.2f} bars  stationary_std={self.ou.stationary_std:.6f}\n"
            f"k_entry={self.k_entry:.3f}  k_exit={self.k_exit:.3f}\n"
            f"ADF p-value={self.diagnostics['adf']['p_value']:.4f}  "
            f"Hurst={self.diagnostics['hurst']:.3f}  tradeable={self.diagnostics['tradeable']}\n"
            f"win_prob={self.win_prob:.3f}  payoff_ratio={self.payoff_ratio:.3f}"
        )


## 11. Optional: Bloomberg Intraday Tick/Bar Loader (`blpapi`)

Requires the Bloomberg Desktop API running locally (`bbcomm`) alongside your Terminal, or a B-PIPE connection. Skip this cell if you don't have `blpapi` installed — it's only needed if you want to pull live data directly instead of using your own CSV/feed.

In [ ]:
def load_bloomberg_intraday(ticker: str, event_type: str, start, end, interval_min: int = 1,
                             host: str = "localhost", port: int = 8194) -> pd.DataFrame:
    """
    Pull intraday bar data from a local Bloomberg Terminal / B-PIPE session via
    blpapi. Requires the Bloomberg Desktop API running (bbcomm) or a B-PIPE
    connection. Returns a DataFrame with ['price','volume'] indexed by time.

    ticker: e.g. "AAPL US Equity"
    event_type: "TRADE", "BID", "ASK", etc.
    start/end: datetime objects
    """
    try:
        import blpapi
    except ImportError as e:
        raise ImportError(
            "blpapi is not installed. Install from Bloomberg's package index:\n"
            "pip install blpapi --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/"
        ) from e

    session_opts = blpapi.SessionOptions()
    session_opts.setServerHost(host)
    session_opts.setServerPort(port)
    session = blpapi.Session(session_opts)

    if not session.start():
        raise ConnectionError("Failed to start Bloomberg session — is the Terminal/bbcomm running?")
    if not session.openService("//blp/refdata"):
        raise ConnectionError("Failed to open //blp/refdata service.")

    service = session.getService("//blp/refdata")
    request = service.createRequest("IntradayBarRequest")
    request.set("security", ticker)
    request.set("eventType", event_type)
    request.set("interval", interval_min)
    request.set("startDateTime", start)
    request.set("endDateTime", end)

    session.sendRequest(request)

    rows = []
    while True:
        ev = session.nextEvent(500)
        for msg in ev:
            if msg.hasElement("barData"):
                bars = msg.getElement("barData").getElement("barTickData")
                for i in range(bars.numValues()):
                    bar = bars.getValueAsElement(i)
                    rows.append({
                        "time": bar.getElementAsDatetime("time"),
                        "price": bar.getElementAsFloat("close"),
                        "volume": bar.getElementAsInteger("volume"),
                    })
        if ev.eventType() == blpapi.Event.RESPONSE:
            break

    session.stop()
    df = pd.DataFrame(rows).set_index("time")
    return df


## 12. Example: Load Data

**Replace this cell** with your real data — either `load_bloomberg_intraday(...)` for live Terminal data, or your own CSV/feed loaded into a DataFrame with a DatetimeIndex and `price`/`volume` columns.

The synthetic example below uses a small grid purely so the notebook runs in a reasonable time. For production, widen `tau_grid`/`n_grid` and raise the Monte Carlo `n_sims`/`max_steps` in `solve_thresholds` / `estimate_win_prob_and_payoff` for more precision.

**Tuning how visible the price/VWAP gap is:** since `price = base_price * exp(dev)` and VWAP tracks a smoothed version of `base_price`, the gap you see in the plot is driven almost entirely by the OU deviation process `dev`, not by `base_price`'s own drift. Its stationary standard deviation is $\sigma/\sqrt{2\theta}$, so:
- **Lower `true_theta`** → slower reversion, longer half-life ($\ln 2/\theta$ bars) → the deviation drifts further before pulling back → wider, slower swings
- **Higher `true_sigma`** → more diffusion per step → wider swings at any reversion speed
- These compound: `theta=0.05, sigma=0.002` gives a ~0.6% typical gap (subtle); `theta=0.01, sigma=0.01` gives a ~7% typical gap (clearly visible) — used below
- A **wider VWAP window** (`tau`) also exaggerates the visual lag, since VWAP smooths over more history and reacts more slowly to the same price swings

In [ ]:
# --- OPTION A: live Bloomberg data (uncomment if you have blpapi + Terminal running) ---
# from datetime import datetime
# df = load_bloomberg_intraday(
#     ticker="AAPL US Equity",
#     event_type="TRADE",
#     start=datetime(2026, 9, 30, 9, 30),
#     end=datetime(2026, 9, 30, 16, 0),
#     interval_min=1,
# )

# --- OPTION B: synthetic example data (for testing the pipeline) ---
rng = np.random.default_rng(0)
n_bars = 1500  # longer sample so a few full reversion cycles are visible
idx = pd.date_range("2026-09-30 09:30", periods=n_bars, freq="1min")

# Tuning the price/VWAP gap: stationary std of the deviation process is
# sigma / sqrt(2*theta) -- lower theta (slower reversion, longer half-life)
# and/or higher sigma (more diffusion) both widen the visible gap, and they
# compound. half_life = ln(2)/theta, in bars.
#   theta=0.05, sigma=0.002 -> ~0.6% typical gap, 14-bar half-life (subtle)
#   theta=0.01, sigma=0.010 -> ~7.1% typical gap, 69-bar half-life (clearly visible)
true_theta, true_mu, true_sigma = 0.01, 0.0, 0.010
dev = np.zeros(n_bars)
for i in range(1, n_bars):
    dev[i] = dev[i-1] + true_theta * (true_mu - dev[i-1]) + true_sigma * rng.standard_normal()

base_price = 100 * np.exp(np.cumsum(0.0001 * rng.standard_normal(n_bars)))
price = base_price * np.exp(dev)
volume = rng.integers(100, 5000, n_bars)

df = pd.DataFrame({"price": price, "volume": volume}, index=idx)
df.tail()


## 13. Configure & Calibrate

`max_holding_halflives` is the entry-aggressiveness dial: it says how many half-lives of patience you're willing to extend before you stop trusting the fitted mean-reversion relationship. Try `0.5` for noticeably more aggressive (lower $Z$) entries, or `2.0`+ for more conservative ones -- the resulting $k^*$ is always solved from the fitted OU parameters, never hardcoded.

In [ ]:
config = StrategyConfig(
    tau_grid=["5min", "15min", "30min"],  # widen for production, e.g. add "60min"
    n_grid=[150, 250],                    # widen for production, e.g. add 390
    dt=1.0 / 390,                    # 1-min bars -> 390 bars/trading day
    cost_bps=1.0,                    # <-- set this to YOUR real round-trip cost estimate
    sigma_target=0.01,               # <-- portfolio vol budget
    kelly_fraction=0.5,              # <-- risk-aversion dial (0,1]
    max_holding_halflives=1.0,       # <-- entry-aggressiveness dial: lower = less conservative Z
)

strat = VWAPMeanReversionStrategy(config)
strat.calibrate(df)
print(strat.summary())


## 14. Generate Signals & Backtest

In [ ]:
results = strat.generate(df)
rets = backtest_returns(results["price"], results["signal"], cost_bps=config.cost_bps)
sharpe = rets.mean() / rets.std() * np.sqrt(252 * 390) if rets.std() > 0 else np.nan
print(f"In-sample annualized Sharpe (illustrative, synthetic data): {sharpe:.2f}")
results.tail(10)


## 15. Plot: Price vs. VWAP, Z-score, Position, and P&L

Two P&L views, kept deliberately distinct so the chart can't quietly mislead you:

- **Raw-signal P&L** -- one unit of exposure every time `signal` is non-zero, net of transaction costs (`rets` from Section 14). Shows what the entry/exit logic itself is worth, independent of position sizing.
- **Kelly-sized P&L** -- the actual backtested `position` column, which already includes vol-targeting and the fractional-Kelly scalar. If the fitted edge doesn't clear costs, `kelly_f` is clipped to zero and this line is correctly flat at zero -- that's Kelly telling you not to trade, not a bug.

The **Sized Position** panel reflects the same `position` column, so it should always agree with the Kelly-sized P&L line (both flat together, or both active together). The per-trade P&L bars below use the raw-signal trades, so you can see which individual entries were winners vs. losers before any sizing is applied.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Raw-signal P&L: unit exposure per signal, net of costs (rets from Section 14).
cum_signal_pnl = rets.cumsum()

# Kelly-sized P&L: the actual position column (vol-targeted + fractional-Kelly).
# If the fitted edge doesn't clear costs, kelly_f is 0 and this is correctly flat.
sized_rets = backtest_returns(results["price"], results["position"], cost_bps=config.cost_bps)
cum_sized_pnl = sized_rets.cumsum()

buy_hold_rets = results["price"].pct_change().reindex(rets.index).fillna(0)
cum_buy_hold = buy_hold_rets.cumsum()

# Per-trade P&L (raw signal): sum returns over each contiguous non-flat signal
# block, so one bar represents one full trade's outcome.
pos_sign = np.sign(results["signal"]).reindex(rets.index).fillna(0)
trade_id = (pos_sign != pos_sign.shift(1)).cumsum()
trade_pnl = rets.groupby(trade_id).sum()
trade_is_flat = pos_sign.groupby(trade_id).first() == 0
trade_pnl = trade_pnl[~trade_is_flat]
trade_end_time = rets.index.to_series().groupby(trade_id).last()[~trade_is_flat]

fig, axes = plt.subplots(5, 1, figsize=(12, 15), sharex=False,
                          gridspec_kw={"height_ratios": [2, 2, 1.2, 2, 1.5]})

axes[0].plot(results.index, results["price"], label="Price")
axes[0].plot(results.index, results["vwap"], label="VWAP", linestyle="--")
axes[0].set_ylabel("Price")
axes[0].legend()
axes[0].set_title("Price vs. VWAP")

axes[1].plot(results.index, results["zscore"], color="tab:orange")
axes[1].axhline(strat.k_entry, color="red", linestyle=":", label="entry")
axes[1].axhline(-strat.k_entry, color="red", linestyle=":")
axes[1].axhline(strat.k_exit, color="green", linestyle=":", label="exit")
axes[1].axhline(-strat.k_exit, color="green", linestyle=":")
axes[1].set_ylabel("Z-score")
axes[1].legend()
axes[1].set_title("Deviation Z-score vs. Fitted Thresholds")

axes[2].plot(results.index, results["position"], color="tab:green")
axes[2].set_ylabel("Position")
axes[2].set_title("Sized Position (vol-targeted + fractional-Kelly)")

axes[3].plot(cum_signal_pnl.index, cum_signal_pnl, label="Raw-signal P&L (unit size)", color="tab:blue")
axes[3].plot(cum_sized_pnl.index, cum_sized_pnl, label="Kelly-sized P&L (actual position)", color="tab:purple")
axes[3].plot(cum_buy_hold.index, cum_buy_hold, label="Buy & hold", color="gray", linestyle="--")
axes[3].axhline(0, color="black", linewidth=0.8)
axes[3].set_ylabel("Cumulative return")
axes[3].legend()
axes[3].set_title("Cumulative P&L: Raw Signal vs. Kelly-Sized vs. Buy & Hold")

colors = np.where(trade_pnl.values >= 0, "tab:green", "tab:red")
axes[4].bar(trade_end_time, trade_pnl.values, color=colors, width=0.6)
axes[4].axhline(0, color="black", linewidth=0.8)
axes[4].set_ylabel("Per-trade P&L")
axes[4].set_title(f"Per-Trade P&L, raw signal ({(trade_pnl > 0).sum()} wins / {(trade_pnl <= 0).sum()} losses)")

plt.tight_layout()
plt.show()

print(f"Kelly fraction applied to sizing: {fractional_kelly_fraction(strat.win_prob, strat.payoff_ratio, config.kelly_fraction):.4f}")
if fractional_kelly_fraction(strat.win_prob, strat.payoff_ratio, config.kelly_fraction) == 0:
    print("-> 0 means the fitted edge (win_prob/payoff_ratio) doesn't clear costs at full Kelly; "
          "position sizing correctly stays flat even though the raw signal is trading.")


## 16. Inspect the Walk-Forward Grid (optional)

In [ ]:
strat.grid_results.sort_values("oos_sharpe", ascending=False)
